# Synonym Substitution: A Black-Box Attack on Text

`attacks/whitebox/01_HotFlip.ipynb` needs full access to the model's gradients. This notebook removes that
assumption entirely, the text equivalent of `attacks/SquareAttack.ipynb` in the companion vision
repo: the model is only ever **queried** for its output probabilities, never differentiated.

The attack follows the two-stage recipe common to TextFooler (Jin et al., 2020) and PWWS (Ren et
al., 2019): first rank words by how much the model's confidence drops when each is individually
removed (**word importance ranking**, itself a query-only, gradient-free operation), then greedily
replace the most important words with WordNet synonyms that flip the prediction while keeping the
sentence grammatically plausible (same part of speech, not a stopword).


In [1]:
import torch
import torch.nn.functional as F
import nltk
from nltk.corpus import wordnet, stopwords
from transformers import AutoTokenizer, AutoModelForSequenceClassification

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)

STOPWORDS = set(stopwords.words('english'))

### Setup: Target Model
Same target as `attacks/whitebox/01_HotFlip.ipynb`, `distilbert-base-uncased-finetuned-sst-2-english`, so
the two notebooks are directly comparable: same model, same threat model dimension (query cost)
made explicit, only the attacker's access level differs.

In [2]:
MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()
LABELS = model.config.id2label

@torch.no_grad()
def predict_proba(text, label_idx):
    """Returns only the scalar probability of `label_idx` - the one piece of information a
    score-based black-box attacker is allowed to see per query."""
    inputs = tokenizer(text, return_tensors='pt')
    probs = F.softmax(model(**inputs).logits, dim=-1)[0]
    return float(probs[label_idx])

@torch.no_grad()
def predict(text):
    inputs = tokenizer(text, return_tensors='pt')
    probs = F.softmax(model(**inputs).logits, dim=-1)[0]
    label_idx = int(torch.argmax(probs))
    return LABELS[label_idx], float(probs[label_idx]), label_idx

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

### Step 1: Word Importance Ranking
For a sentence $w_1, \dots, w_n$, the importance of word $w_i$ is how much the model's confidence
in the true class drops when that word alone is deleted:
$$I(w_i) = P(y \mid w_1, \dots, w_n) - P(y \mid w_1, \dots, w_{i-1}, w_{i+1}, \dots, w_n)$$
This costs exactly $n$ queries (one per word) and requires no gradient, only the ability to ask
"what does the model think of this slightly different sentence?".

In [3]:
def word_importance_ranking(words, true_label_idx):
    queries_used = 0
    base_prob = predict_proba(' '.join(words), true_label_idx)
    queries_used += 1

    importances = []
    for i in range(len(words)):
        if words[i].lower() in STOPWORDS:
            importances.append((i, -1.0))  # never worth touching first
            continue
        without_word = words[:i] + words[i + 1:]
        prob_without = predict_proba(' '.join(without_word), true_label_idx)
        queries_used += 1
        importances.append((i, base_prob - prob_without))

    ranked = sorted(importances, key=lambda x: x[1], reverse=True)
    return [i for i, _ in ranked], queries_used

### Step 2: WordNet Synonym Candidates
For each candidate word (in importance order), we look up WordNet synonyms restricted to the same
part of speech, so a substitution like "good" (adjective) -> "skillful" stays grammatical, unlike
swapping in a noun or verb sense of an ambiguous word.

In [4]:
POS_MAP = {'NN': wordnet.NOUN, 'VB': wordnet.VERB, 'JJ': wordnet.ADJ, 'RB': wordnet.ADV}

def get_synonyms(word, pos_tag, max_candidates=8):
    wn_pos = POS_MAP.get(pos_tag[:2])
    if wn_pos is None:
        return []
    synonyms = set()
    for syn in wordnet.synsets(word, pos=wn_pos):
        for lemma in syn.lemmas():
            candidate = lemma.name().replace('_', ' ')
            if candidate.lower() != word.lower():
                synonyms.add(candidate)
    return list(synonyms)[:max_candidates]

### Putting It Together
Visit words in importance order; at each one, try every WordNet synonym and greedily keep whichever
candidate does the most damage to the true class's probability, exactly the same greedy-acceptance
pattern `attacks/SquareAttack.ipynb` uses, just over a discrete word list instead of pixel patches.

In [5]:
def synonym_attack(text, max_words_to_replace=10):
    words = text.split()
    pos_tags = [tag for _, tag in nltk.pos_tag(words)]
    _, _, true_label_idx = predict(text)

    order, queries_used = word_importance_ranking(words, true_label_idx)
    substitutions = []

    for count, i in enumerate(order):
        if count >= max_words_to_replace:
            break
        current_prob = predict_proba(' '.join(words), true_label_idx)
        queries_used += 1

        best_word, best_prob = words[i], current_prob
        for candidate in get_synonyms(words[i], pos_tags[i]):
            trial_words = words[:i] + [candidate] + words[i + 1:]
            trial_prob = predict_proba(' '.join(trial_words), true_label_idx)
            queries_used += 1
            if trial_prob < best_prob:
                best_word, best_prob = candidate, trial_prob

        if best_word != words[i]:
            substitutions.append((words[i], best_word))
            words[i] = best_word

        adv_text = ' '.join(words)
        _, _, current_label_idx = predict(adv_text)
        if current_label_idx != true_label_idx:
            break

    return ' '.join(words), substitutions, queries_used

### Evaluation
The same four review sentences attacked in `attacks/whitebox/01_HotFlip.ipynb`, so the two threat models can
be compared directly: how many words need to change, and how many model queries each strategy
costs, to reach the same goal.

In [6]:
SAMPLE_REVIEWS = [
    "This movie was absolutely fantastic, I loved every minute of it.",
    "The acting was terrible and the plot made no sense at all.",
    "A brilliant, moving performance that will stay with me for years.",
    "I was bored throughout the entire film and nearly walked out.",
]

for review in SAMPLE_REVIEWS:
    orig_label, orig_prob, _ = predict(review)
    adv_text, substitutions, queries_used = synonym_attack(review)
    adv_label, adv_prob, _ = predict(adv_text)

    print(f"\n{'='*80}")
    print(f"Original ({orig_label}, {orig_prob*100:.1f}%): {review}")
    print(f"Substitutions ({len(substitutions)}): " + ', '.join(f"'{a}'->'{b}'" for a, b in substitutions))
    print(f"Adversarial ({adv_label}, {adv_prob*100:.1f}%): {adv_text}")
    print(f"Success: {adv_label != orig_label} | Queries used: {queries_used}")


Original (POSITIVE, 100.0%): This movie was absolutely fantastic, I loved every minute of it.
Substitutions (5): 'absolutely'->'dead', 'movie'->'film', 'loved'->'sleep with', 'minute'->'mo', 'was'->'cost'
Adversarial (NEGATIVE, 95.2%): This film cost dead fantastic, I sleep with every mo of it.
Success: True | Queries used: 52



Original (NEGATIVE, 100.0%): The acting was terrible and the plot made no sense at all.
Substitutions (6): 'sense'->'gumption', 'terrible'->'wicked', 'acting'->'performing', 'plot'->'secret plan', 'made'->'pee-pee', 'was'->'represent'
Adversarial (NEGATIVE, 97.8%): The performing represent wicked and the secret plan pee-pee no gumption at all.
Success: False | Queries used: 57



Original (POSITIVE, 100.0%): A brilliant, moving performance that will stay with me for years.
Substitutions (3): 'performance'->'carrying into action', 'moving'->'locomote', 'stay'->'delay'
Adversarial (POSITIVE, 99.8%): A brilliant, locomote carrying into action that will delay with me for years.
Success: False | Queries used: 38



Original (NEGATIVE, 100.0%): I was bored throughout the entire film and nearly walked out.
Substitutions (5): 'bored'->'drill', 'film'->'picture', 'nearly'->'intimately', 'entire'->'intact', 'walked'->'take the air'
Adversarial (POSITIVE, 99.9%): I was drill throughout the intact picture and intimately take the air out.
Success: True | Queries used: 40
